In [ ]:
import sys
import os
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    !pip install -U ipython
    !git clone https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git

    repo_url = "https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git"
    repo_path = "/content/QML_for_radar_classification"
    colab_run_dir = os.path.join(repo_path, 'colab_run')

    def run(cmd, cwd=None):
        return subprocess.check_output(cmd, cwd=cwd, text=True).strip()

    if not os.path.isdir(os.path.join(repo_path, ".git")):
        run(["git", "clone", repo_url, repo_path])
    else:
        run(["git", "fetch", "origin"], cwd=repo_path)
        local_head = run(["git", "rev-parse", "HEAD"], cwd=repo_path)
        remote_head = run(["git", "rev-parse", "origin/HEAD"], cwd=repo_path)
        if local_head != remote_head:
            run(["git", "reset", "--hard", "origin/HEAD"], cwd=repo_path)

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)


    os.makedirs(colab_run_dir, exist_ok=True)
    os.chdir(colab_run_dir)

    !pip install -q pennylane
    !pip install "ray[tune]"


%load_ext autoreload
%autoreload 2

%aimport -torch
%aimport -numpy
%aimport -qiskit
%aimport -pennylane
%aimport -ray
%aimport -sklearn

# These import from Data folder are necessary for pickle load to work
from Data.Primitives.environment_classes import Drone, Radar, Context
from Data.Primitives.noise_models import AdditiveWhiteGaussianNoise
from Data.Primitives.presets import *
from Data.Generators.synthetic_dataset_generator import DatasetMetadata, DataRequest

from MachineLearning.Processing.file_loader import SyntheticDataFileLoader
from MachineLearning.Processing.frequency_domain_parser import FrequencyDomainDataParser
from MachineLearning.Processing.time_domain_parser import TimeDomainDataParser
from MachineLearning.Torch_datasets.synthetic_time_dataset import SyntheticTimeDomainRadarDataset
from MachineLearning.Torch_datasets.synthetic_frequency_dataset import SyntheticFrequencyDomainRadarDataset
from MachineLearning.Models.experiment_pure.classical_neural_network import ClassicalNeuralNetwork
from MachineLearning.Models.experiment_pure.quantum_neural_network import QuantumNeuralNetwork
from MachineLearning.Processing.data_visualizer import DataVisualizer
from MachineLearning.Trainers.statistical_trainer import TrainerForModelStatistics
from MachineLearning.Trainers.hyperparameter_trainer import TrainerForHyperparameterSearch

from matplotlib import pyplot as plt
import numpy as np

from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import sympy

import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
from torch.optim import SGD

import ray
from ray import tune
from ray.tune import Checkpoint
from ray.tune.schedulers import ASHAScheduler

MODEL_REGISTRY = {
    "ClassicalNeuralNetwork": ClassicalNeuralNetwork,
}

### Checking preprocessing functionalities

In [ ]:
PROJECT_ROOT = Path().cwd().parent
type = "time_domain"
load_path = PROJECT_ROOT / "Data" / "Datasets" / type / "training_dataset.pkl"

md = DatasetMetadata.create_from_path(load_path)
synt_dataset = SyntheticDataFileLoader(dataset_metadata=md)

obj = synt_dataset.peek_sample(index=6000)

td_data_parser = TimeDomainDataParser()
signal, label, misc_data = td_data_parser.extract_training_data_and_label(obj)
td_data_parser.plot_drone_spectrogram(signal, misc_data, nperseg=32, noverlap=16)

In [ ]:
parsed_signal, label, misc_data = td_data_parser.parse_data_object(obj)
parsed_signal, label, misc_data

In [ ]:
PROJECT_ROOT = Path().cwd().parent
type = "frequency_domain"
load_path = PROJECT_ROOT / "Data" / "Datasets" / type / "training_dataset.pkl"

md = DatasetMetadata.create_from_path(load_path)
synt_dataset = SyntheticDataFileLoader(dataset_metadata=md)

obj = synt_dataset.peek_sample(index=6000)

td_data_parser = FrequencyDomainDataParser()
signal, label, misc_data = td_data_parser.parse_data_object(obj, bin_size=1, return_mag=False)
print(signal.shape)

### Hyperparameter learner

In [2]:
# Safely import the colab module
try:
    from google.colab import output
    if 'google.colab' in sys.modules:
        IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = Path().cwd().parent
ray.shutdown()

ctx = ray.init(
    _metrics_export_port=8080,
    runtime_env={
        "working_dir": str(PROJECT_ROOT),
        "excludes": [
            "Data/Datasets"
        ]
    }
)

if IN_COLAB:
    print("Loading Ray Dashboard:")
    output.serve_kernel_port_as_iframe(8265, height=600)

    print("Loading Ray Metrics Export:")
    output.serve_kernel_port_as_iframe(8080, height=400)
else:
    print(f"Running locally. Dashboard available at: {ctx.dashboard_url}")
    print("Metrics available at: http://127.0.0.1:8080")

2026-04-21 21:48:31,682	INFO worker.py:2014 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-04-21 21:48:32,871	INFO packaging.py:588 -- Creating a file package for local module 'E:\Zycie\Nauka\Studia\Inf\Praca dyplomowa\Thesis_project\QML_for_radar_classification'.
2026-04-21 21:48:32,956	WARNING packaging.py:430 -- File E:\Zycie\Nauka\Studia\Inf\Praca dyplomowa\Thesis_project\QML_for_radar_classification\.git\objects\31\9db0d1654352e4db09aece7c618a7d83cc7308 is very large (14.43MiB). Consider adding this file to the 'excludes' list to skip uploading it: `ray.init(..., runtime_env={'excludes': ['E:\Zycie\Nauka\Studia\Inf\Praca dyplomowa\Thesis_project\QML_for_radar_classification\.git\objects\31\9db0d1654352e4db09aece7c618a7d83cc7308']})`
2026-04-21 21:48:33,489	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_80f4974b767ea0e5.zip' (33.00MiB) to Ray cluster...
2026-04-21 21:48:33,626	INFO packaging.py:393 -- Successfully pushed file packa

Running locally. Dashboard available at: 127.0.0.1:8265
Metrics available at: http://127.0.0.1:8080


C:\Users\adria\miniconda3\envs\Qiskit_workshop\Lib\site-packages\ray\_private\worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(train_model pid=5396) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/adria/ray_results/train_model_2026-04-21_21-48-43/t_1a3bf_00000/checkpoint_000000)
(train_model pid=5396) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/adria/ray_results/train_model_2026-04-21_21-48-43/t_1a3bf_00000/checkpoint_000001)
(train_model pid=5396) Checkpoint successfully created at: Checkpoint(filesystem=local, path=C:/Users/adria/ray_results/train_model_2026-04-21_21-48-43/t_1a3bf_00000/checkpoint_000002)
(train_model pid=5396) Checkpoint successfully created at: Checkpoint(filesystem=lo

In [3]:
config_params = 32
divs_array = sympy.divisors(config_params)

pair_map = {div : config_params // div for div in divs_array}

config = {
    "layers": tune.grid_search(list(pair_map.keys())),
    "neurons_per_layer": tune.sample_from(lambda config: pair_map[config["layers"]]),
    "lr": tune.loguniform(1e-5, 1e-3),
    "batch_size": tune.choice([2, 4, 8, 16]),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    'epochs' : 250
}


max_num_epochs = 250
num_trials =  10
scheduler = ASHAScheduler(
    max_t=max_num_epochs,
    grace_period=20,
    reduction_factor=2,
)

cpus_per_trial = 10
gpus_per_trial = 0

PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"
testing_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "testing_dataset.pkl"

trainer = TrainerForHyperparameterSearch(
    training_path=training_path,
    validating_path=validating_path,
    testing_path=testing_path,
    criterion = nn.CrossEntropyLoss()
)

tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(trainer.train_model, model_class=ClassicalNeuralNetwork),
        resources={"cpu": cpus_per_trial, "gpu": gpus_per_trial}
    ),
    tune_config=tune.TuneConfig(
        metric="accuracy",
        mode="max",
        scheduler=scheduler,
        num_samples=num_trials,
        trial_dirname_creator=lambda trial: f"t_{trial.trial_id}"
    ),
    param_space=config,
)
results = tuner.fit()

# Current best trial: 8f0ab_00008 with loss=0.4411230981349945 and params={'layers': 1, 'neurons_per_layer': 128, 'lr': 9.763374597090547e-05, 'batch_size': 4, 'device': 'cuda', 'epochs': 250}
# Current best trial: 7a17b_00064 with loss=0.43200549483299255 and params={'layers': 2, 'neurons_per_layer': 32, 'lr': 0.000471644, 'batch_size': 2, 'device': 'cpu', 'epochs': 160}
# 32 params batch size 8	layers 1	0.000148743	250	81.1576	0.493999	0.746429

2026-04-21 21:54:25,997	ERROR tune_controller.py:1331 -- Trial task failed for trial train_model_1a3bf_00003
Traceback (most recent call last):
  File "C:\Users\adria\miniconda3\envs\Qiskit_workshop\Lib\site-packages\ray\air\execution\_internal\event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "C:\Users\adria\miniconda3\envs\Qiskit_workshop\Lib\site-packages\ray\_private\auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adria\miniconda3\envs\Qiskit_workshop\Lib\site-packages\ray\_private\client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adria\miniconda3\envs\Qiskit_workshop\Lib\site-packages\ray\_private\worker.py", line 2972, in get
    values, debugger_breakpoint = worker.get_objects(
                                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\adria\miniconda3

### Statistical learner

In [2]:
def l1_normalize_1d(x):
    return normalize(x, p=1, dim=0)

def l2_normalize_1d(x):
    return normalize(x, dim=0)

In [4]:
PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"
testing_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "testing_dataset.pkl"


config = {
    "layers": 1,
    "neurons_per_layer": 128,
    "lr": 9.763374597090547e-05,
    "batch_size": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

trainer = TrainerForModelStatistics(training_path, validating_path, testing_path, criterion = nn.CrossEntropyLoss())
data_array_all_runs = trainer.train_model(ClassicalNeuralNetwork, config, number_of_trials=5, number_of_epochs=500)

Model runs: 100%|██████████| 5/5 [54:34<00:00, 654.95s/it]


In [ ]:
data_array = torch.tensor(
    [data_array_all_runs[i]['accuracy'] for i in range(len(data_array_all_runs))],
    dtype=torch.float32
)

plotter = DataVisualizer()
means, stds = plotter.calculate_statistics(data_array)
plotter.plot_statistics(means, stds)

### Quantum Neural Network

In [ ]:
batch_size = 64
num_qubits = 10

PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"

training_data = SyntheticTimeDomainRadarDataset(training_path)
validating_data = SyntheticTimeDomainRadarDataset(validating_path, mean=training_data.mean, std=training_data.std)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
validating_dataloader = DataLoader(validating_data, batch_size=64, shuffle=True)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
quantum_model = QuantumNeuralNetwork(num_qubits).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(quantum_model.parameters(), lr=1e-3)

trainer = TrainerForHyperparameterSearch(
    training_dataloader=train_dataloader,
    validating_dataloader=validating_dataloader,
    testing_dataloader=None,
    loss_fn=criterion,
)

epochs = 10
accuracy_array=[]
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    trainer.train()
    acc = trainer.test()
    accuracy_array.append(acc)
print("Done!")